# 1-3 画質改善技術：再構成フィルタ、逐次近似再構成、ノイズ除去

範囲1 医用画像の種類と原理｜補足

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/Medical_Imaging_Seminar_Public/blob/main/hiroshima/1_modalities/1-3_image_quality.ipynb)

## この回で分かるようになること

- 画像のノイズ・分解能・コントラストの関係と、それを表す指標（SD、コントラスト、RMSE）を説明できる
- 再構成フィルタと画像フィルタで、ノイズと鮮鋭さが引き換えになることを説明できる
- ハイブリッド型逐次近似再構成と、モデルベース型（フル）逐次近似再構成の違いを説明できる
- ウェーブレットによるノイズ除去と、ディープラーニングによるノイズ除去・再構成のしくみと注意点を説明できる

## 使うデータ

- 自作の数値ファントム（水の楕円に、骨・脂肪・軟部組織、低コントラストの病変、細い棒を入れた模型）。ダウンロードは不要です

## 0. 準備

**Colab で開いた場合**: 上の「Open In Colab」ボタンから開き、次のセル（Colab用の準備）を実行します。ウェーブレットによるノイズ除去（2-5）で使う `PyWavelets` を追加でインストールします。そのあと、残りのセルを上から順に実行します。

**VS Code で開いた場合**: ノートブック右上の「カーネルの選択」で `.venv` を選んでから、次のセルを実行します。Colab用の準備セルは、そのまま実行しても何も起きません（`uv sync` で環境が整っているため）。

In [ ]:
import sys
import subprocess

# Colab では、ウェーブレットによるノイズ除去に使う PyWavelets を用意する
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
    print("準備できました")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from skimage.transform import radon, iradon, iradon_sart
from skimage.restoration import denoise_tv_chambolle, denoise_wavelet, estimate_sigma

## 1. 解説

### 1-1. 画質を表す指標と、線量との関係

1-1のノートブック（1-6）で見たとおり、CTの画像のノイズは、検出器に届く光子の数のばらつきから生じ、その大きさ（SD）は線量（mAs）の平方根に反比例します。ノイズを半分にするには、線量を4倍にしなければなりません。画質改善技術は、同じ線量でノイズを減らすための技術です。言い換えると、同じノイズのまま線量を減らすための技術でもあります。

画質は、ノイズだけでは決まりません。この回では、次の3つの指標を使います。

- **ノイズ（SD）**：構造のない一様な部分（水）で測ったCT値の標準偏差です。小さいほどざらつきが少なくなります
- **コントラスト**：病変と周りのCT値の差です。模型の低コントラストの病変は、本来は周りより+20 HU高く作ってあります。ノイズを減らす処理で病変までぼけると、この差が小さくなります。コントラストをノイズで割った値を、コントラスト対ノイズ比（CNR）と呼びます
- **RMSE**：真の模型との差の2乗平均の平方根です。シミュレーションでは真の値がわかるので計算できますが、実際の患者さんの画像では計算できません

もう1つの大事な性質が空間分解能（細かい構造をどこまで見分けられるか）です。模型の右下にある細い棒の並びが、見分けられるかどうかで確かめます。ノイズを減らす処理の多くは、細かい構造や小さく淡い病変も一緒に鈍らせます。「ノイズが減った」ことと「病変が見やすくなった」ことは同じではありません。

### 1-2. 再構成フィルタと画像フィルタ

#### 再構成フィルタ（再構成関数、カーネル）

1-1のノートブック（1-6）で見たとおり、フィルタ補正逆投影法（FBP）では、逆投影のぼけを打ち消すために、投影データに高い空間周波数を強めるフィルタ（ランプフィルタ）をかけます。ところが、ノイズも高い空間周波数の成分を多く含むので、ランプフィルタはノイズも強めてしまいます。そこで実際の装置では、ランプフィルタの高い周波数の側を少し弱めたフィルタを、目的に合わせて使い分けます。このフィルタを、再構成関数やカーネルと呼びます。

![FBP with different reconstruction filters](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/iq_fbp_kernels.png)

左は3つのフィルタの周波数特性です。ランプフィルタは周波数に比例して強め続けますが、Shepp-Loganフィルタは高い周波数でやや抑え、Hannフィルタは高い周波数をほとんど通しません。右の3枚は、同じ投影データ（この図だけ、ほかの図の4倍の線量）を3つのフィルタで再構成したものです。ノイズのSDは45 HU、36 HU、17 HUと下がりますが、輪郭がぼけるのでRMSEは大きくなります。

臨床では、同じ撮影データから、肺や骨を見るための鋭いカーネルの画像と、腹部の臓器を見るための滑らかなカーネルの画像を別々に作ることがよくあります。どちらが良いかではなく、見たいものによって選びます。

#### 画像フィルタ

再構成が終わった画像に、フィルタをかけてノイズを減らすこともできます。

- **ガウシアンフィルタ**：各画素を、周りの画素との重み付き平均で置き換えます。近い画素ほど重みを大きくします
- **メディアンフィルタ**：各画素を、周りの画素の値の中央値で置き換えます。飛び抜けた値に強く、輪郭が平均ほどにはぼけにくい性質があります

![Gaussian and median filters applied to the FBP image](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/iq_image_filters.png)

低線量のFBP画像（SD 101 HU）に、ガウシアンフィルタとメディアンフィルタをかけた例です。どちらもSDは20 HU前後まで下がりますが、右のプロファイル（細い棒の並びを横切る線上のCT値）では、ガウシアンフィルタで4本の棒が1つのかたまりになり、見分けられなくなっています。病変のコントラストも、本来の+20 HUから+16 HU、+12 HUに下がっています。

MRIでも、k空間の周辺を弱めるフィルタ（1-2のノートブックの2-3で見た、中心だけを残す操作を緩やかにしたもの）が、同じ引き換えの関係でノイズを減らすのに使われます。

出典: 自作（NumPy と scikit-image によるシミュレーション、matplotlib による作図。生成スクリプト: `instructor/make_figures_1_3_iq.py`。`uv run python instructor/make_figures_1_3_iq.py` で作り直せます）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-3. 逐次近似再構成

FBPは、投影データから画像を1回の計算で求める方法で、「ノイズがない」「X線の焦点は点で、検出器の素子は無限に小さい」などの理想的な条件を仮定しています。実際のデータはこの仮定からずれるので、線量を下げるとノイズやアーチファクトが目立ちます。逐次近似再構成（iterative reconstruction、IR）は、画像を少しずつ直しながら、測った投影データに合う画像を探す方法です。

![The loop of iterative reconstruction](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/iq_iterative_loop.png)

1. 仮の画像（FBPの画像や、何もない画像）から出発します
2. 仮の画像をX線が通ったら、どんな投影データになるかを計算します（順投影）。このとき、装置の幾何、焦点の大きさ、検出器の性質などを取り入れたモデル（システムモデル）を使えます
3. 計算した投影データと、実際に測った投影データを比べます
4. 光子の数が少なかった経路のデータは、ばらつきが大きく信頼できないので、比べるときの重みを小さくします（統計モデル、ノイズモデル）
5. 差を画像に戻して（逆投影して）、仮の画像を直します
6. 隣り合う画素の細かいばらつきを抑え、はっきりした輪郭は残すような制約（正則化）をかけます
7. 2に戻って繰り返します

式で書くと、逐次近似再構成は、次の量を最小にする画像 $x$ を探しています。

$$\sum_i w_i \left(p_i - [A x]_i\right)^2 + \beta\, R(x)$$

$p_i$ は測った投影値、$[Ax]_i$ は画像 $x$ から計算した投影値、$w_i$ は経路ごとの重み（光子の数が多いほど大きい）、$R(x)$ は画像のざらつきの大きさ、$\beta$ はデータとの一致と滑らかさのどちらを重んじるかを決める係数です。

#### ハイブリッド型逐次近似再構成

ハイブリッド型は、FBPを土台にして、投影データと画像のそれぞれで、統計モデルを使った反復的なノイズ除去を行う方法です。1回の順投影と逆投影を何十回も繰り返すことはしないので、計算はFBPとほぼ同じ速さで終わります。仕上がりの画像では、処理した画像とFBPの画像を決まった割合で混ぜ、その割合（ブレンド率）で、ノイズの減り方と見慣れた画像の質感のつり合いを調整します。2009年ごろから各社の装置に搭載され、現在は日常の撮影に広く使われています（例：ASiR、iDose⁴、AIDR 3D）。

#### モデルベース型（フル）逐次近似再構成

モデルベース型は、上の流れのとおり、詳しいシステムモデルと統計モデルを使って、順投影と逆投影を何度も繰り返す方法です（例：Veo、IMR、FIRST）。ノイズが大きく減るだけでなく、焦点や検出器の大きさによるぼけもモデルで補正するので、空間分解能が上がり、ストリーク状のアーチファクトも減ります。そのかわり、計算に時間がかかります。また、ノイズの質感がFBPと大きく変わり、塗り絵のように平坦な画像に見えることがあります。線量を下げすぎると、小さく淡い病変が見えにくくなる点にも注意が必要です。

![Comparison of FBP, hybrid IR and model-based IR](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/iq_ir_compare.png)

同じ低線量の投影データから、FBP、ハイブリッド型、モデルベース型を簡単なモデルで再現して再構成した例です。ハイブリッド型は「光子の少ない経路ほど投影データを滑らかにする → FBP → 画像の反復的なノイズ除去 → FBPの画像と混ぜる」、モデルベース型は「SART（順投影と逆投影で画像を直す反復法）と、輪郭を残す正則化（全変動、TV）の繰り返し」で代用しています。実際の製品とは細部が違います。

- FBP：SDは101 HUで、低コントラストの病変（橙の破線の円）はノイズに埋もれています
- ハイブリッド型：SDは7 HUまで下がりますが、病変のコントラストは+12 HUに下がり、細い棒もぼけています
- モデルベース型：SDは15 HUで、病変のコントラストは+20 HUとほぼ保たれ、RMSEは最も小さくなっています。一方で、画像がまだら模様になっています

SDだけを比べるとハイブリッド型が最も良く見えますが、病変の見えやすさでは逆の結果になっています。画質改善技術を評価するときは、ノイズだけでなく、病変のコントラストや細かい構造、ノイズの質感もあわせて確かめる必要があります。数値は、処理の強さ（パラメータ）によって大きく変わります。

出典: 自作（NumPy と scikit-image によるシミュレーション、matplotlib による作図。生成スクリプト: `instructor/make_figures_1_3_iq.py`。`uv run python instructor/make_figures_1_3_iq.py` で作り直せます）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-4. ウェーブレットによるノイズ除去

1-2のノートブックで見たフーリエ変換は、画像全体に広がる波の足し合わせで画像を表しました。ウェーブレット変換は、短い範囲にだけ広がる小さな波（ウェーブレット）を、大きさと位置を変えて並べ、画像を表します。そのため、「どこに」「どのくらいの細かさの」構造があるかを同時に表せます。

2次元のウェーブレット変換では、画像を、粗い画像（近似）と、横・縦・斜めの細かい変化（詳細係数）に分けます。粗い画像をさらに同じように分けることを繰り返すと、細かさの段階ごとの成分が得られます。

![Wavelet decomposition and soft thresholding](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/iq_wavelet.png)

- 2枚目：2段階のウェーブレット変換の係数です。左上の小さな画像が粗い近似で、ほかの部分が詳細係数です。模型の輪郭は、少数の大きな係数として現れています
- 3枚目：詳細係数の値の分布です。大部分は0の近くに集まった小さな値で、その多くがノイズです。輪郭などの構造は、少数の大きな値になります
- 4枚目：しきい値（赤い破線）より小さい係数を0にし、大きい係数もしきい値の分だけ0に近づけてから（ソフトしきい値処理）、逆変換で画像に戻した結果です。SDは45 HUから11 HUに下がっています（この図は、ほかの図の4倍の線量）

しきい値は、最も細かい斜めの係数からノイズの大きさを見積もって決めるのが一般的です。しきい値を大きくするほどノイズは減りますが、淡い構造も消え、ブロック状の不自然な模様が出ることがあります。「画像は少数の大きな係数で表せる」というウェーブレットの性質は、MRIの圧縮センシング（1-2のノートブックの1-7）でも使われています。

出典: 自作（NumPy と scikit-image によるシミュレーション、matplotlib による作図。生成スクリプト: `instructor/make_figures_1_3_iq.py`。`uv run python instructor/make_figures_1_3_iq.py` で作り直せます）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-5. ディープラーニングによるノイズ除去と再構成

ディープラーニングを使う方法では、低線量の画像と、それに対応する高画質の画像の組をたくさん用意し、畳み込みニューラルネットワーク（CNN。範囲5で扱います）に「低線量の画像から高画質の画像を予測する」ことを学習させます。学習では、予測と正解の差（損失）が小さくなるように、ネットワークの中の多数の重みを少しずつ調整します。

使い方には、次のようなものがあります。

- **画像のノイズ除去**：FBPなどで作った画像に、学習したネットワークをかけます
- **深層学習再構成（DLR）**：再構成の過程にネットワークを組み込み、高線量の撮影やモデルベース型逐次近似で作った画像を正解として学習させます。CTでは AiCE、TrueFidelity、Precise Image などが臨床で使われています
- **MRIの再構成**：間引いて集めたk空間のデータから、ノイズの少ない画像を作ります（1-2のノートブックの1-7）

学習が済めば計算は速く、モデルベース型に比べて、FBPに近い見慣れた質感を保ちやすいとされています。

![Training a small CNN denoiser and its output](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/iq_deep_learning.png)

位置や大きさを変えた300個の模型で、7層の小さなCNNを学習させた例です。評価には、学習に使っていない、本文のほかの図と同じ模型を使っています。

- 左：学習が進むにつれて損失が下がっています
- 出力：SDは101 HUから8 HUまで下がり、骨や脂肪の円の輪郭ははっきり残っています
- ただし、低コントラストの病変のコントラストは+20 HUから+11 HUに下がり、小さい2つの病変はほとんど消えています

この例は小さなネットワークと少ないデータによるもので、実際の製品よりずっと単純です。それでも、ディープラーニングの方法に共通する注意点がよく表れています。ネットワークは「学習したデータにありそうな、もっともらしい画像」を出力するので、学習データと違う条件（装置、線量、体格、まれな病変）では、実際にはある構造を消したり、ない構造を作ったりすることがあります（ハルシネーション）。臨床で使うときは、見たい病変が実際に見えるかを確かめる評価が欠かせません。読影する側も、どの再構成法で作った画像かを知っておく必要があります。

出典: 自作（NumPy と scikit-image によるシミュレーション、matplotlib による作図。生成スクリプト: `instructor/make_figures_1_3_iq.py`。`uv run python instructor/make_figures_1_3_iq.py` で作り直せます）／ 作者: 本教材の作成者 ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-6. 手法のまとめ

| 方法 | どこで処理するか | 主な長所 | 主な注意点 |
|---|---|---|---|
| 再構成フィルタ（カーネル） | FBPの中 | 速い。目的に合わせて鋭さを選べる | ノイズと鮮鋭さが引き換え |
| 画像フィルタ（ガウシアン、メディアン） | 再構成のあとの画像 | 簡単で速い | 細かい構造や淡い病変も鈍る |
| ハイブリッド型逐次近似 | 投影データと画像（FBPが土台） | FBPに近い速さでノイズを減らせる。質感を調整できる | 強くかけると淡い病変のコントラストが下がる |
| モデルベース型逐次近似 | 順投影と逆投影を繰り返す再構成 | ノイズが大きく減り、分解能も上がる | 計算が重い。質感が変わる |
| ウェーブレットによるノイズ除去 | 画像（ウェーブレットの係数） | 輪郭を残しやすい | しきい値が大きいと淡い構造が消え、不自然な模様が出る |
| ディープラーニング | 画像、または再構成の中 | 学習後は速い。質感を保ちやすい | 学習データと違う条件で、構造を消したり作ったりすることがある |

MRIのノイズは、光子の数のばらつきではなく、主に体や受信回路の熱による電気的な雑音から生じます。それでも、k空間のフィルタ、ウェーブレットを使う圧縮センシング、ディープラーニングによる再構成といった考え方は、CTと共通です。

## 2. 動かしてみる

教員が用意したコードを、上から順に実行します。

### 2-1. 模型を作り、低線量のCTを模擬する

水の楕円に、骨・脂肪・軟部組織、低コントラストの病変（+20 HU）、細い棒を入れた模型を作ります。投影データを作り、検出器に届く光子の数のばらつき（ポアソン分布）を加えてから、FBPで再構成します。

In [ ]:
N = 128              # 画像の1辺の画素数
PIX = 25.0 / N       # 1画素の大きさ（cm）。撮影範囲を 25 cm とする
MU_WATER = 0.2       # 水の線減弱係数（1/cm）
theta = np.linspace(0.0, 180.0, 180, endpoint=False)  # 投影の角度（度）
yy, xx = (np.mgrid[:N, :N] - N / 2 + 0.5) * PIX       # 各画素の位置（cm）


def disk(cx, cy, r):
    """中心 (cx, cy)、半径 r（cm）の円の内側を True にする"""
    return (xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2


# 水の楕円に、骨・脂肪・軟部組織と、+20 HU の低コントラストの病変3つを入れる
mu = np.where((xx / 10.5) ** 2 + (yy / 9.0) ** 2 < 1, MU_WATER, 0.0)
for cx, cy, r, value in [(-5, -3.5, 1.5, 0.40), (5, -3.5, 1.5, 0.18), (0, -5.5, 1.5, 0.22),
                         (-4, 3.5, 0.9, 0.204), (-0.8, 3.5, 0.6, 0.204), (2.4, 3.5, 0.4, 0.204)]:
    mu[disk(cx, cy, r)] = value
# 右下に細い棒を4本並べる（空間分解能を見るため）
for k in range(4):
    mu[(np.abs(yy - 6.0) < 1.2) & (np.abs(xx - (4.5 + k * 0.6)) < 0.15)] = 0.3


def to_hu(x):
    """線減弱係数を CT値（HU）に変換する"""
    return 1000 * (x - MU_WATER) / MU_WATER


roi = disk(0, 1, 1.5)                                  # ノイズを測る、水の一様な部分
lesion = disk(-4, 3.5, 0.6)                            # 一番大きい病変の中心部
background = disk(-4, 3.5, 2.0) & ~disk(-4, 3.5, 1.3)  # 病変の周り


def measure(x):
    """ノイズ（SD）、病変と周りの差（本来は +20 HU）、真の模型との差（RMSE）を返す"""
    h = to_hu(x)
    sd = h[roi].std()
    contrast = h[lesion].mean() - h[background].mean()
    rmse = np.sqrt(np.mean((h - to_hu(mu))[mu > 0] ** 2))
    return sd, contrast, rmse


# 投影データを作り、光子の数のばらつきを加える
rng = np.random.default_rng(0)
N0 = 1.0e4                                   # 1本の経路あたりに入射する光子の数（線量に比例）
p_true = radon(mu, theta) * PIX              # 経路に沿った μ × 厚さ
counts = rng.poisson(N0 * np.exp(-p_true))   # 検出器に届く光子の数
p = -np.log(np.maximum(counts, 1) / N0)      # 装置が再構成に使う投影値


def fbp(p, filter_name="ramp"):
    """フィルタ補正逆投影法で再構成し、線減弱係数（1/cm）で返す"""
    return iradon(p, theta, filter_name=filter_name) / PIX


results = {}                                 # 手法ごとの画像をためておく
results["FBP (ramp)"] = fbp(p)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
axes[0].imshow(to_hu(mu), cmap="gray", vmin=-150, vmax=150)
axes[0].set_title("True object")
axes[1].imshow(p, cmap="gray", aspect="auto")
axes[1].set_title("Noisy sinogram")
axes[2].imshow(to_hu(results["FBP (ramp)"]), cmap="gray", vmin=-150, vmax=150)
axes[2].set_title("FBP (low dose)")
for ax in (axes[0], axes[2]):
    ax.axis("off")
plt.tight_layout()
plt.show()

print("FBP: SD %.1f HU, 病変 %+.1f HU, RMSE %.1f" % measure(results["FBP (ramp)"]))

### 2-2. 再構成フィルタを変える

同じ投影データを、ランプ、Shepp-Logan、Hannの3つのフィルタで再構成します。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, name in zip(axes, ["ramp", "shepp-logan", "hann"]):
    img = fbp(p, name)
    results[f"FBP ({name})"] = img
    sd, contrast, rmse = measure(img)
    ax.imshow(to_hu(img), cmap="gray", vmin=-150, vmax=150)
    ax.set_title(f"FBP, {name}\nSD {sd:.0f} HU, lesion {contrast:+.0f} HU")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 2-3. 画像フィルタをかける

ランプフィルタで再構成した画像に、ガウシアンフィルタとメディアンフィルタをかけます。細い棒を横切る線の上のCT値（プロファイル）も比べます。

In [ ]:
base = results["FBP (ramp)"]
results["Gaussian filter"] = gaussian_filter(base, sigma=1.2)  # 周りとの重み付き平均
results["Median filter"] = median_filter(base, size=5)         # 周り 5×5 画素の中央値

names = ["FBP (ramp)", "Gaussian filter", "Median filter"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
for ax, name in zip(axes, names):
    sd, contrast, rmse = measure(results[name])
    ax.imshow(to_hu(results[name]), cmap="gray", vmin=-150, vmax=150)
    ax.set_title(f"{name}\nSD {sd:.0f} HU, lesion {contrast:+.0f} HU")
    ax.axis("off")

row = int(N / 2 + 6.0 / PIX)          # 細い棒を通る行
x_cm = xx[row]
axes[3].plot(x_cm, to_hu(mu)[row], color="0.6", label="true")
for name in names:
    axes[3].plot(x_cm, to_hu(results[name])[row], label=name)
axes[3].set_xlim(2.5, 8.0)
axes[3].set_xlabel("Position (cm)")
axes[3].set_ylabel("CT number (HU)")
axes[3].set_title("Profile across thin bars")
axes[3].legend(fontsize=8)
plt.tight_layout()
plt.show()

### 2-4. 逐次近似再構成（SARTとTVの繰り返し）

`iradon_sart` は、仮の画像から計算した投影と、測った投影の差を逆投影して画像を直す反復法（SART）の1回分を行う関数です。そのあと、輪郭を残してノイズを抑える正則化（`denoise_tv_chambolle`、全変動＝TV）をかけることを、15回繰り返します。1-3のモデルベース型逐次近似を、統計モデルを省いて簡単にしたものです。数秒かかります。

In [ ]:
x = np.zeros((N, N))                  # 何もない画像から出発する
for i in range(15):
    x = iradon_sart(p / PIX, theta, image=x, relaxation=0.25)  # 投影データに合うように画像を直す
    x = denoise_tv_chambolle(x, weight=0.025)                   # 正則化：輪郭を残してノイズを抑える
results["Iterative (SART + TV)"] = x

sd, contrast, rmse = measure(x)
plt.figure(figsize=(4.5, 4.5))
plt.imshow(to_hu(x), cmap="gray", vmin=-150, vmax=150)
plt.title(f"SART + TV, 15 iterations\nSD {sd:.0f} HU, lesion {contrast:+.0f} HU")
plt.axis("off")
plt.show()

### 2-5. ウェーブレットでノイズを除く

`estimate_sigma` で画像のノイズの大きさを見積もり、`denoise_wavelet` でウェーブレットの係数にソフトしきい値処理をかけます。

In [ ]:
h = to_hu(results["FBP (ramp)"])
sigma = estimate_sigma(h)             # 最も細かいウェーブレット係数から、ノイズの大きさを見積もる
h_w = denoise_wavelet(h, sigma=sigma, wavelet="db2", mode="soft", method="VisuShrink", rescale_sigma=False)
results["Wavelet denoising"] = h_w / 1000 * MU_WATER + MU_WATER   # CT値を線減弱係数に戻す

sd, contrast, rmse = measure(results["Wavelet denoising"])
print(f"見積もったノイズの大きさ: {sigma:.1f} HU")
plt.figure(figsize=(4.5, 4.5))
plt.imshow(h_w, cmap="gray", vmin=-150, vmax=150)
plt.title(f"Wavelet denoising\nSD {sd:.0f} HU, lesion {contrast:+.0f} HU")
plt.axis("off")
plt.show()

### 2-6. 指標を表にして比べる

ここまでの手法の指標を並べます。

In [ ]:
print(f"{'method':24s} {'SD (HU)':>8s} {'lesion (HU)':>12s} {'RMSE':>7s}")
for name, img in results.items():
    sd, contrast, rmse = measure(img)
    print(f"{name:24s} {sd:8.1f} {contrast:+12.1f} {rmse:7.1f}")

## 3. AIに頼んでみる

頼む前に、ソース管理（`Ctrl+Shift+G`）でここまでの状態をコミットします。AIが変更した後は、差分を見て採用するかどうかを決めます。頼み方は `setup/03_ai_assistant.md` を参照してください。

発展課題は、基本課題を終えた人が取り組みます。

### 基本課題1：線量を変える

入射する光子の数 `N0` を 2.5e3、1e4、4e4 の3通りに変えて、FBP（ランプ）と逐次近似（SART + TV）でそれぞれ再構成し、SD・病変のコントラスト・RMSEを表にするよう頼みます。線量を4倍にしたとき、FBPのSDがおよそ何倍になるかも確かめます。

### 基本課題2：違いを聞く

ハイブリッド型逐次近似再構成とモデルベース型逐次近似再構成の違いを、解説役（`tutor`）に質問します。答えを読んで、1-3の解説と食い違う点がないか確かめ、振り返りに書きます。

### 発展課題1：正則化の強さを変える

2-4の `weight`（TVの強さ）を 0.005 から 0.08 まで何通りか変え、SDと病変のコントラストの関係を1つのグラフに描くよう頼みます。ノイズを減らすほど、病変のコントラストがどうなるかを考えます。

### 発展課題2：小さなCNNでノイズを除く

PyTorchで小さなCNNを作り、位置や大きさを変えた模型の低線量FBP画像から、ノイズのない画像を予測するように学習させるよう頼みます。評価には、学習に使っていないこの回の模型を使います。範囲5（CNN）の予習です。CPUでは学習に数分かかることがあります。

## 4. AIの答えを確かめる

確認できた項目は `[ ]` を `[x]` に書き換えます。

- [ ] SDを測る領域が、構造のない水の一様な部分になっている
- [ ] 手法を比べるとき、どれも同じ投影データ（同じ乱数で作ったもの）を使っている
- [ ] ノイズ（SD）だけでなく、病変のコントラストと細い棒の見え方も確認した
- [ ] 線量を4倍にしたときのFBPのSDの変化が、1/√4 = 0.5倍に近いことを確かめた
- [ ] 発展課題2で、学習に使う模型と評価に使う模型を分けている

## 5. 振り返り

| 項目 | 記入欄 |
|---|---|
| 使ったプロンプト | |
| AIの答えで直した点・採用しなかった点 | |
| この回で分かったこと | |
| まだ分からないこと | |

記入したら保存してコミットします。提出のしかたは授業で指示します。